In [ ]:
# start by importing the requests library
import requests
# import the pprint library to make the output more readable
from pprint import pprint
# import pandas for easier quantile calculations and formatting
import pandas as pd
# import scipy for quantile calculations
from scipy.stats.mstats import mquantiles

# define the api-endpoint
API_ENDPOINT = "https://api.water.noaa.gov/hefs"
# define location id
LOCATION_ID = "PGRC2"
# define parameter id
PARAMETER_ID = "QINE"

  ### Retrieving Data from the API

We first import Python's requests library, which allows us to make HTTP requests. We then define the API endpoint we'll be interacting with, setting API_ENDPOINT to https://api.water.noaa.gov/hefs/. Here's how we do it:

### Most Recent Ensemble Forecast for a Location

In other cases, you might want to retrieve for the quantile calculation. The API allows ordering by a certain value of a parameter. This is achieved by adding `ordering=` to a parameter name.

For example, to retrieve the latest QINE ensemble forecast for location id PGRC2, you'll specify `location_id=PGRC2`, `parameter_id=QINE`, `ordering=-start_date_date`, and `limit=1`.  Note that the negative sign in front of start_date_date tells the API to sort the results by start_date_date descending

`/v1/headers/?location_id={LOCATION_ID}&parameter_id={PARAMETER_ID}&ordering=-start_date_date&limit=1`

Where LOCATION_ID and PARAMETER_ID are the location id and parameter id of the request respectively.

In [ ]:
from datetime import datetime, timedelta
# create a series request with location_id, parameter_id, and ordering filters
uri = API_ENDPOINT + f"/v1/headers/?location_id={LOCATION_ID}&parameter_id={PARAMETER_ID}&ordering=-start_date_date&limit=1"
# get the response
response = requests.request("GET", uri)
# get most recent start date
startDate = response.json()['results'][0]['start_date_date']
# print the response
pprint(response.json())

Next, with the most recent forecast startDate, we will retrieve all of the ensemble data for the given start_date_date.

We will use the term `limit` to set the total series output to 50 (there should should be around 33 ensembles). To retrieve the ensemble members, you can format the URI like this:

`/v1/ensembles/?location_id={LOCATION_ID}&parameter_id={PARAMETER_ID}&forecast_date_date={startDate}&limit=50`

In [ ]:
# create a series request with location_id, parameter_id, start_date_date, and limit filters
uri = API_ENDPOINT + f"/v1/ensembles/?location_id={LOCATION_ID}&parameter_id={PARAMETER_ID}&start_date_date={startDate}&limit=50"
# get the response
response = requests.request("GET", uri)
# print the response
pprint(response.json())

We now use the data above to calculate the quantiles for each forecast. First we need to format the data into a dataframe with corresponding datetimes to enable the quantile calculations:

In [ ]:
from os import times
from ast import Index
import matplotlib.pyplot as plt
import datetime


# get json formatted version of the response
responseJ = response.json()

# check if results is present the response data
if 'results' in responseJ and responseJ['results']:

  # dictionary to different values per timestep
  step_vals = {}
  # array of timesteps
  timesteps = []
  # iterate through result in results
  for result in responseJ['results']:
    # check if event is present in response data
    if 'events' in result and result['events']:
      # iterate through each item in events
      for i in result['events']:
          # format datetime
          datetime_str = f"{i['date']} {i['time']}"
          # check if step vals has a dictionary key that is datetime
          if(not step_vals.get(datetime_str)):
            timesteps.append(datetime_str)
            step_vals[datetime_str] = []
          # add value to stepvals array for corresponding datetime
          step_vals[datetime_str].append(float(i['value']))
  # create dataframe from stepvals
  csDF = pd.DataFrame.from_dict(step_vals, orient='index')
  # sort dataframe by datetime
  csDF.sort_index(inplace=True)
  # display dataframe
  print(csDF)

Now, we use the scipy mQuantiles function to calculate the quantiles from the data frame created above.



In [ ]:
# list of different quantile p values to use
quantile_list = [.95,.9,.8,.75,.7,.6,.5,.4,.3,.25,.2,.1,.05]
# calculate quantiles from values of the results
csMQ = mquantiles(csDF.dropna(axis=1),quantile_list,0,0,axis=1)
# iterate through length of mquantile response
for i in range(len(csMQ)):
  # display the timestep
  print(f"Quantiles for {timesteps[i]}: ")
  #print all quantiles for the given timestep
  for j in range(len(csMQ[i])):
    print(f"p{quantile_list[j]}: {csMQ[i][j]}")
  print()

print("\n")
for i in range(len(step_vals)):
  # find min csDF value
  dfMIN = min(step_vals[timesteps[i]])
  print(f"Min from {timesteps[i]}= {dfMIN}")
  # find max csDF value
  dfMAX = max(step_vals[timesteps[i]])
  print(f"Max from {timesteps[i]}= {dfMAX}\n")





Finally, we plot the different timestep quantiles by value for different exceedance probabilities (inverse of the p value).

In [ ]:
import matplotlib.pyplot as plt
# iterate through length of mquantile response
for i in range(len(timesteps)):
  # check if timestep has been converted to datetime
  if type(timesteps[i]) != datetime.datetime:
    # convert timestep to datetime
    timesteps[i] = datetime.datetime.strptime(timesteps[i], '%Y-%m-%d %H:%M:%S')
for i in range(len(quantile_list)):
  plt.plot(timesteps,csMQ[:,i])
# reverse quantile list to show exceedance probability instead
reverse = quantile_list[::-1]

# prepare for putting the legend outside of graph
ax = plt.subplot(111)
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
plt.legend(reverse, title="Exceedance Probability",loc="center left", fontsize="small",bbox_to_anchor=(1, 0.5))
# set x graph label
plt.xlabel('Time')
# set y graph label
plt.ylabel('Value')
# set graph title
plt.title('Quantiles for Different Probabilities')
# rotate x text vertically for better visibility
plt.xticks(rotation='vertical')
# display the actaul plot
plt.show()

## Summary

In this notebook, we learned how to use the HEFS API to retrieve the latest ensemble forecast for a given location. We also learned how to calculate quantiles for the respective forecasts.

These techniques can be used to retrieve, filter, and paginate data for all HEFS ensemble forecasts.